# 08 — Web-GIS (pengganti ArcGIS Online / .slpk)

1. `TerrainLayer`: permukaan 3D dari tile elevasi AWS Terrain (terrarium) dengan tekstur citra Esri World Imagery,
   di-stream langsung dari server tile (tanpa API key, tanpa menyebar data mentah).
2. `PolygonLayer` gedung `extruded=True`, `get_elevation=Bld_Height`, warna per kategori, tooltip atribut
   (pengganti popup identify). pydeck tidak membawa `TerrainExtension`, jadi tiap poligon diberi koordinat z =
   tinggi terrain web di bawahnya (`terrarium_z`, sampel dari tile yang sama) agar gedung tepat berdiri di tanah.
3. Danau: poligon datar (median terrain di tepinya + 1 m), digambar selalu di atas terrain (`depthCompare`).
4. Opsional `ScenegraphLayer` memuat `ui_lod2.glb` (sudah membawa terrain sendiri, jadi `TerrainLayer` dimatikan).
5. `deck.to_html` → `output/ui3d_web.html`, satu file yang bisa dibuka di browser atau dihosting statis.

Terrain web (SRTM ~30 m) lebih kasar dari DEMNAS, dan tile terrarium hanya sampai zoom 15, sehingga tekstur citra\nikut zoom 15 (~4,8 m/piksel). Versi tajam (citra 0,3 m) ada di `ui_lod2.glb` (notebook 07/11). Bila LOD2 diaktifkan, buka lewat
server lokal (`python -m http.server` di folder `output/`), karena browser memblokir `.glb` dari `file://`.

In [1]:
import json

import geopandas as gpd
import pandas as pd
import pydeck as pdk

from ui3d_common import CATEGORY_COLORS, out, terrain_layer, web_credit, web_ground, web_rings

ADD_LOD2 = False  # True = ganti terrain + gedung PolygonLayer dengan ScenegraphLayer ui_lod2.glb

gdb = out("ui3d.gpkg")
bld = gpd.read_file(gdb, layer="buildings").to_crs(4326).explode(index_parts=False)
water = gpd.read_file(gdb, layer="water").to_crs(4326).explode(index_parts=False)
water = water[water.geom_type == "Polygon"]

# Dasar gedung = titik terendah terrain di bawah footprint, agar tidak melayang di lereng.
b = pd.DataFrame(bld.drop(columns="geometry"))
b["z0"] = web_ground(bld.geometry, 0)
b["polygon"] = [web_rings(g, z) for g, z in zip(bld.geometry, b.z0)]
b["color"] = b.category.map(CATEGORY_COLORS)
b["name"] = b["name"].fillna("").replace("", "(tanpa nama)")
w = pd.DataFrame(water.drop(columns="geometry"))
w["polygon"] = [web_rings(g, z + 1) for g, z in zip(water.geometry, web_ground(water.geometry, 50))]
print(f"Gedung: {len(b)}, danau: {len(w)}; dasar gedung {b.z0.min():.0f}–{b.z0.max():.0f} m (terrain web)")

Gedung: 995, danau: 19; dasar gedung 54–88 m (terrain web)


In [2]:
layers = [
    # SRTM menyimpan kanopi di tepi danau, jadi muka air datar sering tertutup terrain. depthCompare "always":
    # air selalu digambar di atas terrain (relief Depok landai, tidak ada bukit yang seharusnya menutupinya).
    pdk.Layer("PolygonLayer", w, get_polygon="polygon", get_fill_color=[45, 95, 135, 230], stroked=False,
              parameters={"depthCompare": "always"}),
    pdk.Layer("PolygonLayer", b, id="buildings", get_polygon="polygon", extruded=True, get_elevation="Bld_Height",
              get_fill_color="color", get_line_color=[40, 40, 40], pickable=True, auto_highlight=True),
]

if ADD_LOD2:
    o = json.load(open(out("origin.json")))
    layers = [pdk.Layer("ScenegraphLayer", [{"position": [o["lon"], o["lat"], 0]}], scenegraph="ui_lod2.glb",
                        get_position="position", get_orientation=[0, 0, 90], size_scale=1, _lighting="pbr")]
else:
    layers.insert(0, terrain_layer())

c = bld.union_all().centroid
deck = pdk.Deck(
    layers=layers,
    initial_view_state=pdk.ViewState(longitude=c.x, latitude=c.y, zoom=15.6, pitch=55, bearing=-20),
    map_provider=None,
    description=web_credit(),
    tooltip={"html": "<b>{name}</b><br/>Kategori: {category}<br/>Lantai: {levels} ({N_source})<br/>"
                     "Tinggi: {Bld_Height} m<br/>Elevasi dasar: {base_elev} m<br/>Footprint: {footprint_source}"},
)
html = out("ui3d_web.html")
deck.to_html(str(html), open_browser=False)
print("Ditulis:", html)
deck

Ditulis: /home/luwes-system/Desktop/bahan ajar 2026/ui3d/output/ui3d_web.html


{
  "description": "<style>.deck-json-description-box{z-index:10!important;left:0!important;bottom:0!important;top:auto!important}</style><div style='font:11px sans-serif;background:#fffc;padding:2px 6px'>Citra: Esri World Imagery (pendidikan/non-komersial) \u00b7 Terrain: AWS Terrain Tiles (Mapzen, SRTM)</div>",
  "initialViewState": {
    "bearing": -20,
    "latitude": -6.3632481623242345,
    "longitude": 106.82740489533633,
    "pitch": 55,
    "zoom": 15.6
  },
  "layers": [
    {
      "@@type": "TerrainLayer",
      "elevationData": "https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png",
      "elevationDecoder": {
        "bScaler": 0.00390625,
        "gScaler": 1,
        "offset": -32768,
        "rScaler": 256
      },
      "id": "terrain",
      "maxZoom": 15,
      "meshMaxError": 1,
      "texture": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
    },
    {
      "@@type": "PolygonLayer",
      "data": [
        {
          "id": 28994172,
          "name": "Danau Kenanga",
          "natural": "water",
          "polygon": [
            [
              [
                106.8294416,
                -6.365912199999994,
                75.60546875
              ],
              [
                106.8293086,
                -6.366036700000004,
                75.60546875
              ],
              [
                106.8292093,
                -6.366143400000005,
                75.60546875
              ],
              [
                106.829133,
                -6.366272699999993,
                75.60546875
              ],
              [
                106.8290825,
                -6.366399499999999,
                75.60546875
              ],
              [
                106.8290544,
                -6.3665991999999925,
                75.60546875
              ],
              [
                106.8290323,
                -6.366955499999996,
                75.60546875
              ],
              [
                106.8292576,
                -6.367070999999999,
                75.60546875
              ],
              [
                106.8295231,
                -6.367222999999992,
                75.60546875
              ],
              [
                106.8298075,
                -6.367358899999998,
                75.60546875
              ],
              [
                106.8300532,
                -6.367497499999999,
                75.60546875
              ],
              [
                106.8301548,
                -6.367568799999998,
                75.60546875
              ],
              [
                106.8302856,
                -6.367746100000009,
                75.60546875
              ],
              [
                106.8304807,
                -6.367949399999996,
                75.60546875
              ],
              [
                106.8305665,
                -6.367938699999993,
                75.60546875
              ],
              [
                106.8306631,
                -6.367913400000004,
                75.60546875
              ],
              [
                106.83078110000001,
                -6.3678894,
                75.60546875
              ],
              [
                106.8308642,
                -6.367846700000007,
                75.60546875
              ],
              [
                106.8309554,
                -6.367734800000005,
                75.60546875
              ],
              [
                106.8311017,
                -6.367412100000006,
                75.60546875
              ],
              [
                106.8314244,
                -6.366689600000003,
                75.60546875
              ],
              [
                106.8315281,
                -6.366412599999995,
                75.60546875
              ],
              [
                106.8312418,
                -6.366285699999999,
      